In [ ]:
import pandas as pd

df = pd.read_pickle('../data/clean/MachineLearning_Clean.pkl')
df.head()

In [ ]:
df['HasClaim'] = df['TotalClaims'] > 0
df['ClaimSever'] = df['TotalClaims'].where(df['ClaimFreq'], None)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']




In [ ]:
province_risk = df.groupby('Province')['HasClaim'].mean()

province_risk

C:\Users\ruthg\AppData\Local\Temp\ipykernel_4336\3531241414.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  province_risk = df.groupby('Province')['HasClaim'].mean()


Province
Eastern Cape     0.001648
Free State       0.001358
Gauteng          0.003356
KwaZulu-Natal    0.002845
Limpopo          0.002698
Mpumalanga       0.002428
North West       0.002436
Northern Cape    0.001254
Western Cape     0.002166
Name: HasClaim, dtype: float64

In [19]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['Province'], df['HasClaim'])

chi2, p, dof, expected = chi2_contingency(contingency)


In [22]:
print("chi", chi2, "p", p, "dof", dof, 'expected', expected)

chi 104.19088107029361 p 5.925510718204678e-19 dof 8 expected [[3.02514315e+04 8.45684803e+01]
 [8.07642220e+03 2.25777994e+01]
 [3.92767012e+05 1.09798802e+03]
 [1.69307697e+05 4.73303044e+02]
 [2.47667640e+04 6.92359829e+01]
 [5.25710366e+04 1.46963382e+02]
 [1.42887555e+05 3.99445010e+02]
 [6.36221430e+03 1.77856970e+01]
 [1.70319867e+05 4.76132587e+02]]


In [23]:
province_risk.sort_values(ascending=False)

Province
Gauteng          0.003356
KwaZulu-Natal    0.002845
Limpopo          0.002698
North West       0.002436
Mpumalanga       0.002428
Western Cape     0.002166
Eastern Cape     0.001648
Free State       0.001358
Northern Cape    0.001254
Name: HasClaim, dtype: float64

In [25]:
claim_risk_province = df.groupby('Province')['ClaimSever'].mean()
claim_risk_province.sort_values(ascending=False)

C:\Users\ruthg\AppData\Local\Temp\ipykernel_4336\1741315907.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  claim_risk_province = df.groupby('Province')['ClaimSever'].mean()


Province
Free State       32265.661085
KwaZulu-Natal    29609.487473
Western Cape     28095.849881
Eastern Cape     27128.533277
Gauteng          22243.878396
North West       16963.467035
Mpumalanga       15979.553421
Limpopo          15171.294187
Northern Cape    11186.313596
Name: ClaimSever, dtype: float64

In [30]:
from scipy.stats import f_oneway

claims_only = df[df['HasClaim']]
claim_list = [claims_only[claims_only['Province']==prov]['TotalClaims'] for prov in df['Province'].unique()]

f_stat, p_val = f_oneway(*claim_list)
print(f_stat, p_val)

4.830165899976359 6.304916760425176e-06


In [33]:
groups = [
    group['TotalClaims'].values
    for _, group in claims_only.groupby('PostalCode')
    if len(group) > 5
]
f_stat, p_val = f_oneway(*groups)
print(p_val)

1.3846079972976506e-05


In [41]:
zip_table = pd.crosstab(df['PostalCode'], df['HasClaim'])
print(zip_table)
chi2, p, dof, expected = chi2_contingency(zip_table)
print('chi:', chi2, 'p', p )

HasClaim    False  True 
PostalCode              
1            5329     12
2            1482      6
4              77      0
5             396      4
6             438      2
...           ...    ...
9781          640      3
9830           56      0
9868          100      0
9869         1414      1
9870          220      0

[888 rows x 2 columns]
chi: 1454.46760955055 p 3.152172246339057e-30


In [35]:
zip_margin = df.groupby('PostalCode')['Margin'].mean().sort_values()
print(zip_margin)
gorups = [
    group['Margin'].values
    for _, group in df.groupby('PostalCode')
]
f_stat, p_val = f_oneway(*groups)
print(p_val)

PostalCode
466    -2104.003715
2920   -1613.177313
1342   -1511.886460
1751   -1221.090842
9756   -1125.351747
           ...     
3740     171.417242
3802     172.142169
9744     175.104079
4016     195.716263
3887     196.635975
Name: Margin, Length: 888, dtype: float64
1.3846079972976506e-05


In [ ]:
gender_risk = pd.crosstab(df['Gender'], df['HasClaim'])
chi2, p, dof, expected = chi2_contingency(gender_risk)
print(chi2, p)

13.319739611262976 0.003993781054238153


In [40]:
gender = df.groupby('Gender')['HasClaim'].mean()
gender


C:\Users\ruthg\AppData\Local\Temp\ipykernel_4336\2667111941.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gender = df.groupby('Gender')['HasClaim'].mean()


Gender
Female           0.002073
Male             0.002195
Not specified    0.002833
Unknown          0.001468
Name: HasClaim, dtype: float64

In [43]:
df_gender = df[df['Gender'].isin(['Male', 'Female'])]
gender = df_gender.groupby('Gender')['HasClaim'].mean()
print(gender)

gender_risk = pd.crosstab(df_gender['Gender'], df_gender['HasClaim'])
chi2, p, dof, expected = chi2_contingency(gender_risk)
print(chi2, p)


Gender
Female           0.002073
Male             0.002195
Not specified         NaN
Unknown               NaN
Name: HasClaim, dtype: float64
0.003704891861036439 0.9514644755420456


C:\Users\ruthg\AppData\Local\Temp\ipykernel_4336\1302122792.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gender = df_gender.groupby('Gender')['HasClaim'].mean()
